In [1]:
%matplotlib inline
%load_ext autoreload
%autoreload 2
import sys
sys.path.append('/home/yaroslav/progs/biohack/deeppick_mephi_2026/ml/')

In [2]:
from pathlib import Path
import re
import numpy as np
import pandas as pd
from src.utils import load_image
from tqdm import tqdm

DATA_DIR = Path("/home/yaroslav/progs/biohack/deeppick_mephi_2026/ml/data")

target2id = {"control": 0, "exo": 1, "endo": 2}
tissue2id = {"cortex": 0, "striatum": 1, "cerebellum": 2}
group2id = {"1": 0, "2a": 1, "2b": 2, "3": 3}
side2id = {"none": 0, "left": 1, "right": 2}

BAD_FILES = {
    "cortex_endo_1group_633nm_center2900_obj100_power100_1s_5acc_map35x15_step2_place2.txt",
}


def parse_meta_from_name(name: str):
    stem = Path(name).stem
    parts = stem.split("_")

    tissue = next((t for t in ["cortex", "striatum", "cerebellum"] if t in parts), None)
    target = next((t for t in ["control", "exo", "endo"] if t in parts), None)

    if "left" in parts:
        side = "left"
    elif "right" in parts:
        side = "right"
    else:
        side = "none"

    m_group = re.search(r"_(\d[ABab]?)[Gg]roup", stem)
    if m_group is None:
        raise ValueError(f"Cannot parse group from {name}")
    group = m_group.group(1).lower()

    m_center = re.search(r"center(\d+)", stem)
    if m_center is None:
        raise ValueError(f"Cannot parse center from {name}")
    center = int(m_center.group(1))

    if tissue is None or target is None:
        raise ValueError(f"Cannot parse tissue/target from {name}")

    return {
        "tissue": tissue,
        "target": target,
        "side": side,
        "group": group,
        "center": center,
    }


def fix_orientation(out, x, y):
    # приводим к формату (len(y), len(x), L)
    if out.shape[:2] == (len(y), len(x)):
        return out
    if out.shape[:2] == (len(x), len(y)):
        return out.transpose(1, 0, 2)
    raise ValueError(f"Unexpected shape {out.shape}, len(x)={len(x)}, len(y)={len(y)}")

In [3]:
import numpy as np
from scipy import sparse
from scipy.sparse.linalg import spsolve
from scipy.signal import savgol_filter, find_peaks

def baseline_als(y, lam=1e5, p=0.01, niter=10):
    # ПРИНУДИТЕЛЬНО делаем массив одномерным
    y = np.asarray(y).flatten() 
    
    L = len(y)
    # Матрица разностей второго порядка
    D = sparse.diags([1, -2, 1], [0, 1, 2], shape=(L-2, L))
    # Матрица произведения D.T * D дает размер L x L
    DT_D = lam * D.transpose().dot(D)
    
    w = np.ones(L)
    z = np.zeros(L)
    
    for i in range(niter):
        W = sparse.spdiags(w, 0, L, L)
        # Теперь размеры (L,L) + (L,L) совпадают
        Z = W + DT_D
        z = spsolve(Z, w * y)
        w = p * (y > z) + (1 - p) * (y < z)
    return z

def preprocess_spectrum(intensity, lam=3e3, p=7e-3, polyorder=6, window_length=21):
    """
    Полный пайплайн предобработки одного спектра.
    input: np.array (интенсивности)
    """
    # 1. Удаление базовой линии (ALS)
    # Важно: делаем ПЕРЕД нормализацией
    baseline = baseline_als(intensity, lam=lam, p=p)
    y_no_bg = intensity - baseline
    
    # 2. Сглаживание (Savitzky-Golay)
    # window_length должен быть нечетным. 11-15 обычно хватает.
    y_smooth = savgol_filter(y_no_bg, window_length=window_length, polyorder=polyorder)
    y_noise = y_no_bg - y_smooth  # для анализа шума, если нужно
    
    # 3. Нормализация (L2)
    # Приводим вектор к единичной длине
    norm_l2 = np.linalg.norm(y_smooth)
    y_norm = y_smooth / norm_l2 if norm_l2 > 0 else y_smooth
    
    # 4. Вычисление производных
    # Используем тот же Savitzky-Golay, чтобы не плодить шум
    deriv_1 = savgol_filter(y_norm, window_length=window_length, polyorder=polyorder, deriv=1)
    deriv_2 = savgol_filter(y_norm, window_length=window_length, polyorder=polyorder, deriv=2)
    
    return {
        "clean": y_norm,
        "d1": deriv_1,
        "d2": deriv_2,
        "baseline": baseline,
        "noise": y_noise
    }

In [4]:
img_paths = sorted([
    p for p in DATA_DIR.glob("**/*.txt")
    if "average" not in p.name.lower()
    and p.name not in BAD_FILES
])

records = []

for img_path in tqdm(img_paths):
    meta = parse_meta_from_name(img_path.name)
    out, nu, x, y = load_image(img_path)

    records.append({
        "path": img_path,
        "name": img_path.name,
        "center": meta["center"],
        "target": meta["target"],
        "group": meta["group"],
        "tissue": meta["tissue"],
        "side": meta["side"],
        "nu_min": float(nu.min()),
        "nu_max": float(nu.max()),
        "step_mean": float(np.diff(nu).mean()),
        "step_std": float(np.diff(nu).std()),
        "nu_len": len(nu),
        "raw_shape": tuple(out.shape),
    })

records_df = pd.DataFrame(records)
records_df.head()

  0%|          | 0/236 [00:00<?, ?it/s]

100%|██████████| 236/236 [00:33<00:00,  7.14it/s]


,path,name,center,target,group,tissue,side,nu_min,nu_max,step_mean,step_std,nu_len,raw_shape
0,/home/yaroslav/progs/biohack/deeppick_mephi_20...,cortex_control_1group_633nm_center1500_obj100_...,1500,control,1,cortex,none,926.754883,2002.417969,1.060812,0.061601,1015,"(35, 15, 1015)"
1,/home/yaroslav/progs/biohack/deeppick_mephi_20...,cortex_control_1group_633nm_center1500_obj100_...,1500,control,1,cortex,none,926.754883,2002.417969,1.060812,0.061601,1015,"(35, 15, 1015)"
2,/home/yaroslav/progs/biohack/deeppick_mephi_20...,cortex_control_1group_633nm_center1500_obj100_...,1500,control,1,cortex,none,926.754883,2002.417969,1.060812,0.061601,1015,"(35, 15, 1015)"
3,/home/yaroslav/progs/biohack/deeppick_mephi_20...,cortex_control_1group_633nm_center1500_obj100_...,1500,control,1,cortex,none,926.754883,2002.417969,1.060812,0.061601,1015,"(35, 15, 1015)"
4,/home/yaroslav/progs/biohack/deeppick_mephi_20...,cortex_control_1group_633nm_center1500_obj100_...,1500,control,1,cortex,none,926.754883,2002.417969,1.060812,0.061601,1015,"(35, 15, 1015)"


In [5]:
def resample_cube_to_common_nu(out, nu, common_nu):
    h, w, _ = out.shape
    out_rs = np.empty((h, w, len(common_nu)), dtype=np.float32)

    for i in range(h):
        for j in range(w):
            out_rs[i, j] = np.interp(common_nu, nu, out[i, j])

    return out_rs

In [6]:
def build_dataset_for_center(
    data_dir: Path,
    center_value: int,
    use_side: bool = False,
    preprocess_mode: str = "raw",   # raw | clean | d1 | d2
    preprocess_kwargs: dict | None = None,
):
    # 1) найти все нужные файлы
    img_paths = sorted([
        p for p in data_dir.glob("**/*.txt")
        if "average" not in p.name.lower()
        and p.name not in BAD_FILES
        and f"center{center_value}" in p.name
    ])

    if len(img_paths) == 0:
        raise ValueError(f"No files found for center={center_value}")

    if preprocess_kwargs is None:
        preprocess_kwargs = {}

    if preprocess_mode not in {"raw", "clean", "d1", "d2"}:
        raise ValueError(f"Unknown preprocess_mode={preprocess_mode}")

    # 2) собрать статистику по nu
    mins, maxs, steps = [], [], []

    for img_path in img_paths:
        out, nu, x, y = load_image(img_path)
        mins.append(float(nu.min()))
        maxs.append(float(nu.max()))
        steps.append(float(np.diff(nu).mean()))

    common_min = max(mins)
    common_max = min(maxs)
    common_step = float(np.median(steps))

    common_nu = np.arange(
        common_min,
        common_max + common_step / 2,
        common_step,
        dtype=np.float32
    )

    print(f"center={center_value}")
    print(f"n_files={len(img_paths)}")
    print(f"common range=({common_min:.6f}, {common_max:.6f})")
    print(f"common step={common_step:.9f}")
    print(f"common_nu.shape={common_nu.shape}")
    print(f"preprocess_mode={preprocess_mode}")

    # 3) собрать объекты
    X_num_list = []
    X_cat_list = []
    y_list = []
    groups_list = []
    meta_rows = []
    sample_ids_list = []

    sample_id_counter = 0

    for img_path in tqdm(img_paths):
        meta = parse_meta_from_name(img_path.name)

        out, nu, x, y = load_image(img_path)
        out = fix_orientation(out, x, y)
        out = resample_cube_to_common_nu(out, nu, common_nu)

        # (H, W, L) -> (H*W, L)
        X_map = out.reshape(-1, out.shape[-1]).astype(np.float32)

        # preprocessing по каждому спектру
        if preprocess_mode != "raw":
            X_proc = np.empty_like(X_map, dtype=np.float32)

            for i in range(X_map.shape[0]):
                pp = preprocess_spectrum(X_map[i], **preprocess_kwargs)

                if preprocess_mode == "clean":
                    X_proc[i] = pp["clean"].astype(np.float32)
                elif preprocess_mode == "d1":
                    X_proc[i] = pp["d1"].astype(np.float32)
                elif preprocess_mode == "d2":
                    X_proc[i] = pp["d2"].astype(np.float32)

            X_map = X_proc

        n = X_map.shape[0]

        tissue_id = tissue2id[meta["tissue"]]
        side_id = side2id[meta["side"]]

        if use_side:
            X_cat_map = np.column_stack([
                np.full(n, tissue_id, dtype=np.int64),
                np.full(n, side_id, dtype=np.int64),
            ])
            cat_feature_names = ["tissue_id", "side_id"]
        else:
            X_cat_map = np.column_stack([
                np.full(n, tissue_id, dtype=np.int64),
            ])
            cat_feature_names = ["tissue_id"]

        y_map = np.full(n, target2id[meta["target"]], dtype=np.int64)
        group_map = np.full(n, group2id[meta["group"]], dtype=np.int64)
        sample_ids_map = np.full(n, sample_id_counter, dtype=np.int64)

        X_num_list.append(X_map)
        X_cat_list.append(X_cat_map)
        y_list.append(y_map)
        groups_list.append(group_map)
        sample_ids_list.append(sample_ids_map)

        meta_rows.append({
            "sample_id": sample_id_counter,
            "path": str(img_path),
            "name": img_path.name,
            "n_spectra": n,
            "shape_after": tuple(out.shape),
            "target": meta["target"],
            "group": meta["group"],
            "tissue": meta["tissue"],
            "side": meta["side"],
            "center": meta["center"],
            "nu_min_original": float(nu.min()),
            "nu_max_original": float(nu.max()),
        })

        sample_id_counter += 1

    X_num = np.concatenate(X_num_list, axis=0)
    X_cat = np.concatenate(X_cat_list, axis=0)
    y = np.concatenate(y_list, axis=0)
    groups = np.concatenate(groups_list, axis=0)
    sample_ids = np.concatenate(sample_ids_list, axis=0)

    return {
        "center": center_value,
        "common_nu": common_nu,
        "X_num": X_num,
        "X_cat": X_cat,
        "y": y,
        "groups": groups,
        "sample_ids": sample_ids,
        "meta_df": pd.DataFrame(meta_rows),
        "cat_feature_names": cat_feature_names,
        "preprocess_mode": preprocess_mode,
        "preprocess_kwargs": preprocess_kwargs,
    }

In [7]:
ds_1500 = build_dataset_for_center(
    DATA_DIR,
    center_value=1500,
    use_side=False,
    preprocess_mode="clean",
)
# ds_2900 = build_dataset_for_center(DATA_DIR, center_value=2900, use_side=False, preprocess_mode="clean")

print("1500:", ds_1500["X_num"].shape, ds_1500["X_cat"].shape, ds_1500["y"].shape)
# print("2900:", ds_2900["X_num"].shape, ds_2900["X_cat"].shape, ds_2900["y"].shape)

center=1500
n_files=118
common range=(926.829102, 2002.280273)
common step=1.060815575
common_nu.shape=(1015,)
preprocess_mode=clean


  0%|          | 0/118 [00:00<?, ?it/s]/home/yaroslav/progs/biohack/deeppick_mephi_2026/ml/.venv/lib/python3.11/site-packages/scipy/sparse/_construct.py:543: FutureWarning: Input has data type int64, but the output has been cast to float64.  In the future, the output data type will match the input. To avoid this warning, set the `dtype` parameter to `None` to have the output dtype match the input, or set it to the desired output data type.
Note: In Python 3.11, this warning can be generated by a call of scipy.sparse.diags(), but the code indicated in the warning message will refer to an internal call of scipy.sparse.diags_array(). If that happens, check your code for the use of diags().
  A = diags_array(diagonals, offsets=offsets, shape=shape, dtype=dtype)
/tmp/ipykernel_67969/432346804.py:23: SparseEfficiencyWarning: spsolve requires A be CSC or CSR matrix format
  z = spsolve(Z, w * y)
100%|██████████| 118/118 [05:25<00:00,  2.76s/it]

1500: (61950, 1015) (61950, 1) (61950,)


### Inspect datasets

In [8]:
def inspect_dataset(ds, name="dataset"):
    print(f"=== {name} ===")
    print("X_num:", ds["X_num"].shape)
    print("X_cat:", ds["X_cat"].shape)
    print("y:", ds["y"].shape)
    print("groups:", ds["groups"].shape)
    print()

    print("class counts:", pd.Series(ds["y"]).value_counts().sort_index().to_dict())
    print("group counts:", pd.Series(ds["groups"]).value_counts().sort_index().to_dict())
    print()

    print("target x group:")
    display(pd.crosstab(ds["meta_df"]["group"], ds["meta_df"]["target"]))
    print("tissue x target:")
    display(pd.crosstab(ds["meta_df"]["tissue"], ds["meta_df"]["target"]))

inspect_dataset(ds_1500, "1500")
# inspect_dataset(ds_2900, "2900")

=== 1500 ===
X_num: (61950, 1015)
X_cat: (61950, 1)
y: (61950,)
groups: (61950,)

class counts: {0: 21000, 1: 22050, 2: 18900}
group counts: {0: 9450, 1: 18900, 2: 18900, 3: 14700}

target x group:


target,control,endo,exo
group,,,
1,6,6,6
2a,12,12,12
2b,12,12,12
3,10,6,12


tissue x target:


target,control,endo,exo
tissue,,,
cerebellum,10,6,12
cortex,18,18,18
striatum,12,12,12


In [9]:
def make_catboost_dataframe(ds):
    X_num = ds["X_num"]
    X_cat = ds["X_cat"]

    spec_cols = [f"s_{i}" for i in range(X_num.shape[1])]
    X_df = pd.DataFrame(X_num, columns=spec_cols)

    for i, col in enumerate(ds["cat_feature_names"]):
        X_df[col] = X_cat[:, i].astype(int).astype(str)

    y = ds["y"]
    groups = ds["groups"]
    cat_features = ds["cat_feature_names"]

    return X_df, y, groups, cat_features
    
X1500_df, y1500, g1500, cat1500 = make_catboost_dataframe(ds_1500)
# X2900_df, y2900, g2900, cat2900 = make_catboost_dataframe(ds_2900)
print(X1500_df.shape, len(y1500), cat1500)
# print(X2900_df.shape, len(y2900), cat2900)

(61950, 1016) 61950 ['tissue_id']


### Check cross-validation scheme

In [10]:
from sklearn.model_selection import LeaveOneGroupOut

def inspect_logo_splits(X_df, y, groups, name="dataset"):
    logo = LeaveOneGroupOut()
    print(f"=== LOGO splits for {name} ===")
    for fold, (tr, va) in enumerate(logo.split(X_df, y, groups)):
        print(
            f"fold={fold}, "
            f"train={len(tr)}, val={len(va)}, "
            f"held_out_group={np.unique(groups[va]).tolist()}, "
            f"train_classes={np.unique(y[tr]).tolist()}, "
            f"val_classes={np.unique(y[va]).tolist()}"
        )

inspect_logo_splits(X1500_df, y1500, g1500, "1500")
# inspect_logo_splits(X2900_df, y2900, g2900, "2900")

=== LOGO splits for 1500 ===
fold=0, train=52500, val=9450, held_out_group=[0], train_classes=[0, 1, 2], val_classes=[0, 1, 2]
fold=1, train=43050, val=18900, held_out_group=[1], train_classes=[0, 1, 2], val_classes=[0, 1, 2]
fold=2, train=43050, val=18900, held_out_group=[2], train_classes=[0, 1, 2], val_classes=[0, 1, 2]
fold=3, train=47250, val=14700, held_out_group=[3], train_classes=[0, 1, 2], val_classes=[0, 1, 2]


### Training

In [11]:
from catboost import CatBoostClassifier
from sklearn.metrics import f1_score
from sklearn.model_selection import LeaveOneGroupOut
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt


def run_logo_cv_catboost(
    X_df,
    y,
    groups,
    cat_features,
    verbose=False,
    plot=False,
    metric_name="TotalF1:average=Macro",
):
    logo = LeaveOneGroupOut()
    fold_rows = []

    oof_pred = np.full_like(y, fill_value=-1)
    n_classes = len(np.unique(y))
    oof_proba = np.full((len(y), n_classes), np.nan, dtype=np.float32)

    fold_models = []
    fold_evals = []

    for fold, (tr, va) in enumerate(logo.split(X_df, y, groups)):
        print(f"fold={fold}")

        X_tr = X_df.iloc[tr]
        X_va = X_df.iloc[va]
        y_tr = y[tr]
        y_va = y[va]

        model = CatBoostClassifier(
            loss_function="MultiClass",
            eval_metric=metric_name,
            iterations=400,
            depth=6,
            learning_rate=0.05,
            l2_leaf_reg=3.0,
            random_seed=42,
            verbose=100 if verbose else False,
        )

        model.fit(
            X_tr,
            y_tr,
            cat_features=cat_features,
            eval_set=(X_va, y_va),
            use_best_model=True,
        )

        evals = model.get_evals_result()
        fold_evals.append(evals)
        fold_models.append(model)

        pred = model.predict(X_va).reshape(-1).astype(int)
        proba = model.predict_proba(X_va)

        oof_pred[va] = pred
        oof_proba[va] = proba

        f1m = f1_score(y_va, pred, average="macro")

        best_iteration = model.get_best_iteration()
        best_score = None
        if "validation" in evals and metric_name in evals["validation"]:
            if best_iteration is not None and best_iteration >= 0:
                best_score = evals["validation"][metric_name][best_iteration]

        fold_rows.append({
            "fold": fold,
            "held_out_group": int(np.unique(groups[va])[0]),
            "n_train": len(tr),
            "n_val": len(va),
            "best_iteration": best_iteration,
            "best_val_metric": best_score,
            "f1_macro": f1m,
        })

        if plot:
            val_keys = list(evals.get("validation", {}).keys())
            chosen_metric = metric_name if metric_name in val_keys else (val_keys[0] if len(val_keys) else None)

            if chosen_metric is not None:
                plt.figure(figsize=(8, 4))
                if chosen_metric in evals.get("learn", {}):
                    plt.plot(evals["learn"][chosen_metric], label="train")
                if chosen_metric in evals.get("validation", {}):
                    plt.plot(evals["validation"][chosen_metric], label="val")
                if best_iteration is not None and best_iteration >= 0:
                    plt.axvline(best_iteration, linestyle="--", alpha=0.7, label=f"best_iter={best_iteration}")
                plt.title(f"Fold {fold} | held_out_group={int(np.unique(groups[va])[0])}")
                plt.xlabel("iteration")
                plt.ylabel(chosen_metric)
                plt.grid(True)
                plt.legend()
                plt.show()

    scores_df = pd.DataFrame(fold_rows)
    overall_f1 = f1_score(y, oof_pred, average="macro")

    return scores_df, oof_pred, oof_proba, overall_f1, fold_models, fold_evals

In [12]:
scores_1500, oof_1500, oof_proba_1500, overall_f1_1500, models_1500, evals_1500 = run_logo_cv_catboost(
    X1500_df,
    y1500,
    g1500,
    cat1500,
    verbose=False,
    plot=False,
)

display(scores_1500)
print("OOF macro F1 1500:", overall_f1_1500)

fold=0
fold=1
fold=2
fold=3


,fold,held_out_group,n_train,n_val,best_iteration,best_val_metric,f1_macro
0,0,0,52500,9450,291,0.407111,0.407111
1,1,1,43050,18900,4,0.513372,0.513372
2,2,2,43050,18900,0,0.446236,0.446236
3,3,3,47250,14700,0,0.313113,0.313113


OOF macro F1 1500: 0.4361373989452304


In [14]:
from sklearn.metrics import (
    f1_score,
    classification_report,
    confusion_matrix,
)

def aggregate_map_mean_proba(sample_ids, y_true_pixel, y_proba_pixel):
    n_classes = y_proba_pixel.shape[1]

    df = pd.DataFrame({
        "sample_id": sample_ids,
        "y_true": y_true_pixel,
    })

    for c in range(n_classes):
        df[f"p{c}"] = y_proba_pixel[:, c]

    rows = []
    for sample_id, g in df.groupby("sample_id", sort=True):
        y_true_map = int(g["y_true"].iloc[0])
        mean_proba = g[[f"p{c}" for c in range(n_classes)]].mean(axis=0).values
        y_pred_map = int(np.argmax(mean_proba))

        row = {
            "sample_id": sample_id,
            "y_true": y_true_map,
            "y_pred": y_pred_map,
            "n_pixels": len(g),
        }
        for c in range(n_classes):
            row[f"mean_p{c}"] = float(mean_proba[c])

        rows.append(row)

    return pd.DataFrame(rows)

def aggregate_map_majority_vote(sample_ids, y_true_pixel, y_pred_pixel):
    df = pd.DataFrame({
        "sample_id": sample_ids,
        "y_true": y_true_pixel,
        "y_pred": y_pred_pixel,
    })

    rows = []
    for sample_id, g in df.groupby("sample_id", sort=True):
        y_true_map = int(g["y_true"].iloc[0])
        y_pred_map = int(g["y_pred"].mode().iloc[0])

        rows.append({
            "sample_id": sample_id,
            "y_true": y_true_map,
            "y_pred": y_pred_map,
            "n_pixels": len(g),
        })

    return pd.DataFrame(rows)

id2target = {v: k for k, v in target2id.items()}

def show_eval(y_true, y_pred, title=""):
    print(title)
    print("macro F1:", f1_score(y_true, y_pred, average="macro"))
    print()
    print(classification_report(
        y_true, y_pred,
        target_names=[id2target[i] for i in sorted(id2target)]
    ))
    print("confusion matrix:")
    print(confusion_matrix(y_true, y_pred))

sample_ids_1500 = ds_1500["sample_ids"]

map_df_1500_vote = aggregate_map_majority_vote(
    sample_ids_1500,
    y1500,
    oof_1500,
)

print("MAP-WISE majority vote")
show_eval(
    map_df_1500_vote["y_true"].values,
    map_df_1500_vote["y_pred"].values,
    "1500 map-wise majority vote"
)

map_df_1500_proba = aggregate_map_mean_proba(
    sample_ids_1500,
    y1500,
    oof_proba_1500,
)
print("MAP-WISE mean proba")
show_eval(
    map_df_1500_proba["y_true"].values,
    map_df_1500_proba["y_pred"].values,
    "1500 map-wise mean proba"
)

MAP-WISE majority vote
1500 map-wise majority vote
macro F1: 0.4229339204982072

              precision    recall  f1-score   support

     control       0.29      0.23      0.25        40
         exo       0.76      0.74      0.75        42
        endo       0.24      0.31      0.27        36

    accuracy                           0.43       118
   macro avg       0.43      0.42      0.42       118
weighted avg       0.44      0.43      0.43       118

confusion matrix:
[[ 9  5 26]
 [ 2 31  9]
 [20  5 11]]
MAP-WISE mean proba
1500 map-wise mean proba
macro F1: 0.46253316982256315

              precision    recall  f1-score   support

     control       0.36      0.35      0.35        40
         exo       0.78      0.83      0.80        42
        endo       0.24      0.22      0.23        36

    accuracy                           0.48       118
   macro avg       0.46      0.47      0.46       118
weighted avg       0.47      0.48      0.48       118

confusion matrix:
[[14  6 

In [15]:
map_meta_1500 = ds_1500["meta_df"][["sample_id", "name", "target", "group", "tissue", "side"]].copy()

map_eval_1500 = map_meta_1500.merge(
    map_df_1500_proba[["sample_id", "y_true", "y_pred", "mean_p0", "mean_p1", "mean_p2"]],
    on="sample_id",
    how="left",
)

map_eval_1500["pred_label"] = map_eval_1500["y_pred"].map(id2target)
display(map_eval_1500)

,sample_id,name,target,group,tissue,side,y_true,y_pred,mean_p0,mean_p1,mean_p2,pred_label
0,0,cortex_control_1group_633nm_center1500_obj100_...,control,1,cortex,none,0,2,0.298390,0.176379,0.525231,endo
1,1,cortex_control_1group_633nm_center1500_obj100_...,control,1,cortex,none,0,2,0.351718,0.089523,0.558759,endo
2,2,cortex_control_1group_633nm_center1500_obj100_...,control,1,cortex,none,0,2,0.307014,0.134018,0.558967,endo
3,3,cortex_control_1group_633nm_center1500_obj100_...,control,1,cortex,none,0,2,0.281700,0.131531,0.586769,endo
4,4,cortex_control_1group_633nm_center1500_obj100_...,control,1,cortex,none,0,2,0.192657,0.238814,0.568529,endo
...,...,...,...,...,...,...,...,...,...,...,...,...
113,113,cerebellum_right_exo_3group_633nm_center1500_o...,exo,3,cerebellum,right,1,1,0.329641,0.338519,0.331841,exo
114,114,cerebellum_right_exo_3group_633nm_center1500_o...,exo,3,cerebellum,right,1,1,0.330519,0.339499,0.329982,exo
115,115,cerebellum_right_exo_3group_633nm_center1500_o...,exo,3,cerebellum,right,1,1,0.328218,0.341867,0.329915,exo
116,116,cerebellum_right_exo_3group_633nm_center1500_o...,exo,3,cerebellum,right,1,1,0.331974,0.335912,0.332113,exo
